<a href="https://colab.research.google.com/github/steveonyeke/python-ai-governance/blob/main/phase6-function-calling/01_function_calling_basics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Phase 6: Function Calling Basics
**Goal**: Understand how LLMs use tools and build the
      foundation for tool-use auditing.
**Regulatory mapping**: NIST AI RMF Map and Measure.
**Date**: May 2026
**Status**: In Progress

In [1]:
from google import genai
from google.colab import userdata

client = genai.Client(api_key=userdata.get('GOOGLE_API_KEY'))

response = client.models.generate_content(
    model="gemini-flash-latest",
    contents="Say exactly: Good morning Steve! Phase 6 function calling and tool-use auditing begins."
)
print(response.text)

Good morning Steve! Phase 6 function calling and tool-use auditing begins.


In [2]:
import json
import time
import pandas as pd
from google import genai
from google.colab import userdata, drive
import os

# Setup
drive.mount('/content/drive')
SAVE_PATH = "/content/drive/MyDrive/python-ai-governance/data/"
os.makedirs(SAVE_PATH, exist_ok=True)
client = genai.Client(api_key=userdata.get('GOOGLE_API_KEY'))

# ── DEFINE YOUR TOOLS ──
# These are the functions the LLM can choose to call

def get_ai_regulation_info(jurisdiction: str) -> dict:
    """Returns AI regulation information for a jurisdiction."""
    regulations = {
        "EU": {
            "name": "EU AI Act",
            "status": "In force since August 2024",
            "risk_tiers": 4,
            "max_fine": "35M EUR or 7% annual turnover"
        },
        "USA": {
            "name": "Executive Order on AI",
            "status": "Active since October 2023",
            "risk_tiers": None,
            "max_fine": "No direct fines"
        },
        "UK": {
            "name": "UK AI Safety Institute Framework",
            "status": "Voluntary framework",
            "risk_tiers": None,
            "max_fine": "No direct fines"
        },
        "Ghana": {
            "name": "Ghana AI Policy",
            "status": "Draft stage",
            "risk_tiers": None,
            "max_fine": "Not established"
        }
    }
    return regulations.get(jurisdiction, {"error": f"No data for {jurisdiction}"})

def assess_ai_risk_level(use_case: str, sector: str) -> dict:
    """Assesses the risk level of an AI use case under EU AI Act."""
    high_risk_sectors = ["employment", "credit", "healthcare",
                         "law enforcement", "education", "immigration"]
    prohibited_cases  = ["social scoring", "real-time biometric",
                         "subliminal manipulation"]

    use_case_lower = use_case.lower()
    sector_lower   = sector.lower()

    if any(p in use_case_lower for p in prohibited_cases):
        risk = "PROHIBITED"
        article = "Article 5"
    elif any(s in sector_lower for s in high_risk_sectors):
        risk = "HIGH RISK"
        article = "Annex III and Article 6-49"
    else:
        risk = "LIMITED OR MINIMAL RISK"
        article = "Article 50 or unregulated"

    return {
        "use_case":   use_case,
        "sector":     sector,
        "risk_level": risk,
        "eu_ai_act":  article,
        "requires_compliance": risk in ["PROHIBITED", "HIGH RISK"]
    }

def calculate_potential_fine(violation_type: str,
                              annual_revenue_eur: float) -> dict:
    """Calculates potential EU AI Act fine for a violation."""
    fine_tiers = {
        "prohibited":    {"rate": 0.07, "cap": 35_000_000},
        "high_risk":     {"rate": 0.03, "cap": 15_000_000},
        "information":   {"rate": 0.01, "cap":  7_500_000},
    }

    tier = fine_tiers.get(violation_type.lower())
    if not tier:
        return {"error": f"Unknown violation type: {violation_type}"}

    percentage_fine = annual_revenue_eur * tier["rate"]
    actual_fine     = min(percentage_fine, tier["cap"])

    return {
        "violation_type":    violation_type,
        "annual_revenue":    annual_revenue_eur,
        "percentage_fine":   round(percentage_fine, 2),
        "cap":               tier["cap"],
        "actual_fine":       round(actual_fine, 2),
        "calculation_basis": f"{tier['rate']*100}% of revenue, capped at {tier['cap']:,} EUR"
    }

# ── TOOL REGISTRY ──
# A dictionary mapping tool names to functions
TOOLS = {
    "get_ai_regulation_info":  get_ai_regulation_info,
    "assess_ai_risk_level":    assess_ai_risk_level,
    "calculate_potential_fine": calculate_potential_fine,
}

print("====== TOOL REGISTRY ======")
for name, func in TOOLS.items():
    print(f"Tool: {name}")
    print(f"Description: {func.__doc__}")
    print()
print(f"Total tools available: {len(TOOLS)}")



Mounted at /content/drive
====== TOOL REGISTRY ======
Tool: get_ai_regulation_info
Description: Returns AI regulation information for a jurisdiction.

Tool: assess_ai_risk_level
Description: Assesses the risk level of an AI use case under EU AI Act.

Tool: calculate_potential_fine
Description: Calculates potential EU AI Act fine for a violation.

Total tools available: 3


In [3]:
# - TOOL CALLING ENGINE -

# Define tools in Gemini's required format
gemini_tools = [
    {
        "name": "get_ai_regulation_info",
        "description": "Returns AI regulation information for a jurisdiction.",
        "parameters": {
            "type": "object",
            "properties": {
                "jurisdiction": {
                    "type": "string",
                    "description": "The jurisdiction to look up. Examples: 'EU', 'USA', 'UK', 'Ghana'."
                }
            },
            "required": ["jurisdiction"]
        }
    },
    {
        "name": "assess_ai_risk_level",
        "description": "Assesses the risk level of an AI use case under EU AI Act.",
        "parameters": {
            "type": "object",
            "properties": {
                "use_case": {
                    "type": "string",
                    "description": "Description of the AI use case."
                },
                "sector": {
                    "type": "string",
                    "description": "Industry sector. Examples: employment, healthcare, credit."
                }
            },
            "required": ["use_case", "sector"]
        }
    },
    {
        "name": "calculate_potential_fine",
        "description": "Calculates potential EU AI Act fine for a violation type.",
        "parameters": {
            "type": "object",
            "properties": {
                "violation_type": {
                    "type": "string",
                    "description": "Type of violation. Examples: prohibited, high_risk, or information"
                },
                "annual_revenue_eur": {
                    "type": "number",
                    "description": "Organisation annual revenue in EUR."
                }
            },
            "required": ["violation_type", "annual_revenue_eur"]
        }
    }
]

def call_tool(tool_name, tool_args):
  """Execute a tool from the registry and return results. """
  if tool_name not in TOOLS:
    return {"error": f"Tool {tool_name} not foind in registry"}
  tool_function = TOOLS[tool_name]
  return tool_function(**tool_args)

def ask_with_tools(user_query):
  """Send a query to the LLM with tools available."""
  print(f"\nQuery: {user_query}")
  print("-" * 50)

  # Step 1 - Ask the LLM with tools available
  from google.genai import types
  response = client.models.generate_content(
      model="gemini-flash-latest",
      contents=user_query,
      config=types.GenerateContentConfig(
          tools=[types.Tool(
              function_declarations=[
                  types.FunctionDeclaration(**tool)
                  for tool in gemini_tools
              ]
          )]
      )
  )

  # Step 2- Check if the model wants to call a tool
  tool_calls_made = []

  for part in response.candidates[0].content.parts:
    if hasattr(part, 'function_call') and part.function_call:
      tool_name = part.function_call.name
      tool_args = dict(part.function_call.args)

      print(f"Tool selected: {tool_name}")
      print(f"Parameters:    {json.dumps(tool_args, indent=2)}")

      #Step 3 - Execute the tool
      tool_result = call_tool(tool_name, tool_args)
      print(f"Tool result:   {json.dumps(tool_result, indent=2)}")

      tool_calls_made.append({
          "tool":   tool_name,
          "args":   tool_args,
          "result": tool_result,
      })
 # Step 4 - If no tool was called, return text response
  if not tool_calls_made:
    text = response.candidates[0].content.parts[0].text
    print(f"No tool called. Text response: {text[:200]}")

  return tool_calls_made

# - RUN TEST QUERIES -
test_queries = [
    "What AI regualtions exist in the EU?",
    "I an building an AI  system for CV screening in recruitment. What is my risk level?",
    "A company with 500 million euros annual revenue violated Article 10 of the EU AI Act. What is their fine?",
    "Is using AI for social scoring allowed under the EU AI Act?",
]

all_results = []

for query in test_queries:
  results = ask_with_tools(query)
  all_results.extend(results)
  time.sleep(2)

print("\n======SUMMARY ======")
print(f"Total queries: {len(test_queries)}")
print(f"Total tool calls: {len(all_results)}")

if all_results:
  tools_used = [r["tool"] for r in all_results]
  for tool in TOOLS.keys():
    count = tools_used.count(tool)
    print(f" {tool}: {count} calls")


Query: What AI regualtions exist in the EU?
--------------------------------------------------
Tool selected: get_ai_regulation_info
Parameters:    {
  "jurisdiction": "EU"
}
Tool result:   {
  "name": "EU AI Act",
  "status": "In force since August 2024",
  "risk_tiers": 4,
  "max_fine": "35M EUR or 7% annual turnover"
}

Query: I an building an AI  system for CV screening in recruitment. What is my risk level?
--------------------------------------------------
Tool selected: assess_ai_risk_level
Parameters:    {
  "use_case": "CV screening for recruitment",
  "sector": "employment"
}
Tool result:   {
  "use_case": "CV screening for recruitment",
  "sector": "employment",
  "risk_level": "HIGH RISK",
  "eu_ai_act": "Annex III and Article 6-49",
  "requires_compliance": true
}

Query: A company with 500 million euros annual revenue violated Article 10 of the EU AI Act. What is their fine?
--------------------------------------------------
Tool selected: get_ai_regulation_info
Paramete

## Findings: Function Calling Basics

**Model tested:** gemini-flash-latest.
**Tools available:** 3 governance tools.
**Queries tested:** 4.
**Date:** May 2026

### Results

| Query | Tool Selected | Parameters | Verdict |
|---|---|---|---|
| EU regulations | get_ai_regulation_info | jurisdiction="EU" | Correct |
| CV screening risk | assess_ai_risk_level | sector="employment" | Correct |
| Article 10 fine | calculate_potential_fine | violation_type="high_risk" | Correct |
| Social scoring | assess_ai_risk_level | use_case="social scoring" | Correct |

### Key Findings

1. The model selected the correct tool for every query
   without being told which tool to use. Tool selection
   accuracy: 4 out of 4 (100%).

2. Parameter extraction was accurate. The model correctly
   inferred violation_type="high_risk" from a natural
   language query about Article 10, demonstrating
   semantic understanding of regulatory context.

3. Social scoring correctly returned PROHIBITED under
   Article 5, confirming the model's governance knowledge
   aligns with the correct regulatory classification.

4. Fine calculation finding: companies with annual revenue
   of 500 million euros or more will **always** pay the capped
   15 million euros for high-risk violations, not the
   percentage-based amount.

### Governance Insight
Function calling transforms an LLM from a text generator
into an action-taking system. Every tool call is a
potential governance risk point. Phase 6 Lesson 2 will
deliberately break these tool calls to find those risks.